# 🐼 Notebook 02 · pandas on energy data

Practice for **Part 2** of your visual guide (cards 2.1 → 2.12).

The setup cell creates these tables (no files needed):

| name | what it is |
|---|---|
| `df` | hourly readings for transformers T1–T3 over 14 days: `asset_id`, `ts`, `temp_c`, `load_kw`. It has gaps (NaN) and one temperature spike. |
| `df_raw` | the same, plus 2 duplicated rows (for the data-quality exercise) |
| `failures` | when T1 and T2 failed: `asset_id`, `fail_ts` |
| `assets` | `asset_id`, `age_years`, `region` for T1–T4 |
| `work_orders` | `asset_id`, `cost`, `date`, including one for an unknown asset T9 |

**Rule for every exercise:** don't overwrite `df` itself. Make copies (`df.copy()`) or new variables, so later exercises still get the original.
Start each exercise by *looking*: `df.head()`, `df.info()`, `df.describe()`.

> 💾 **Opened this from GitHub?** Colab won't keep your edits until you choose **File → Save a copy in Drive**. Do that first.

In [ ]:
#@title ▶ Run me first: loads the practice data and the checker { display-mode: "form" }
import math, random, statistics, inspect, functools, csv, io, os, warnings
from collections import Counter, defaultdict, deque
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
_G = globals()
TESTS, DONE = {}, set()

def need(name):
    if name not in _G:
        raise NameError(f"I can't find `{name}`. Run your code cell first, and use exactly that name.")
    return _G[name]

def value(name):
    v = need(name)
    if v is Ellipsis:
        raise NotImplementedError
    return v

def close(a, b, tol=1e-6):
    if a is None:
        raise AssertionError("a result is None. Did you forget `return`?")
    return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))

def expect(fn_name, args, expected, tol=None, **kw):
    f = need(fn_name)
    got = f(*args, **kw)
    shown = ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kw.items()])
    if tol is not None:
        ok = got is not None and close(got, expected, tol)
    else:
        ok = got == expected
    hint = "  (Did you forget `return`?)" if got is None and expected is not None else ""
    assert ok, f"{fn_name}({shown}) should return {expected!r}, but returned {got!r}.{hint}"
    return got

def check(qid):
    try:
        TESTS[qid]()
    except NotImplementedError:
        print(f"✏️  Exercise {qid}: replace the ... with your code, run that cell, then run this check again.")
    except AssertionError as e:
        print(f"❌ Exercise {qid}: not yet. {e}")
    except NameError as e:
        print(f"⚠️  Exercise {qid}: {e}")
    except Exception as e:
        print(f"💥 Exercise {qid}: your code raised {type(e).__name__}: {e}\n   Read the message, fix your cell, re-run it, then check again. (Seeing this before you've written anything is normal.)")
    else:
        DONE.add(qid)
        print(f"✅ Exercise {qid}: all checks passed!")

def progress():
    def key(k):
        return [float(p) if p.replace('.', '', 1).isdigit() else 999 for p in k.lstrip('M').split('.')]
    keys = sorted(TESTS, key=key)
    print(f"{len(DONE)}/{len(TESTS)} exercises passed")
    print("  ".join(("✅ " if k in DONE else "⬜ ") + k for k in keys))

# ---------- practice data ----------
_rng = np.random.default_rng(42)
_idx = pd.date_range("2026-01-01", periods=24 * 14, freq="h")
_n = len(_idx)
df = pd.concat([
    pd.DataFrame({"asset_id": a, "ts": _idx,
                  "temp_c": (60 + 5 * np.sin(np.arange(_n) * 2 * np.pi / 24) + _rng.normal(0, 1, _n)).round(2),
                  "load_kw": (400 + _rng.normal(0, 20, _n)).round(1)})
    for a in ["T1", "T2", "T3"]], ignore_index=True)
for _a, _start, _len in [("T1", 30, 2), ("T1", 100, 6), ("T2", 200, 1), ("T3", 50, 3)]:
    _pos = df.index[df["asset_id"] == _a][_start:_start + _len]
    df.loc[_pos, "temp_c"] = np.nan
df.loc[(df["asset_id"] == "T2") & (df["ts"] == pd.Timestamp("2026-01-10 14:00")), "temp_c"] = 95.0
df_raw = pd.concat([df, df.iloc[[5, 400]]]).sort_values(["asset_id", "ts"], kind="stable").reset_index(drop=True)
failures = pd.DataFrame({"asset_id": ["T1", "T2"], "fail_ts": pd.to_datetime(["2026-01-12 06:00", "2026-01-10 18:00"])})
assets = pd.DataFrame({"asset_id": ["T1", "T2", "T3", "T4"], "age_years": [31, 12, 40, 5], "region": ["coast", "north", "coast", "south"]})
work_orders = pd.DataFrame({"asset_id": ["T1", "T1", "T3", "T3", "T3", "T9"], "cost": [500, 700, 900, 150, 400, 250],
                            "date": pd.to_datetime(["2026-01-03", "2026-02-10", "2026-01-05", "2026-03-01", "2026-03-15", "2026-02-02"])})
_df0, _raw0 = df.copy(), df_raw.copy()

def _ref_fill(d):
    d = d.copy()
    is_na = d["temp_c"].isna()
    run_id = d["temp_c"].notna().groupby(d["asset_id"]).cumsum()
    gap = is_na.groupby([d["asset_id"], run_id]).transform("sum")
    filled = d.groupby("asset_id")["temp_c"].transform(lambda s: s.interpolate(limit_area="inside"))
    d["temp_c"] = filled.where(~(is_na & (gap > 3)))
    return d

def _ref_status(d):
    return np.select([d["temp_c"] > 80, d["temp_c"] > 65], ["critical", "warning"], default="ok")

def _same_df_check():
    assert need("df").shape == _df0.shape, "Looks like `df` itself was changed. Re-run the setup cell, and use df.copy() in your exercises."

# ---------- checks (no peeking needed: solutions are below each exercise) ----------
def _test_2_1():
    _same_df_check()
    assert isinstance(value("temp_col"), pd.Series), "temp_col should be a Series: df['temp_c']"
    ref = _df0[(_df0["asset_id"] == "T2") & (_df0["temp_c"] > 65)]
    got = value("t2_hot")
    assert list(got.index) == list(ref.index), f"t2_hot should have {len(ref)} rows (T2 and temp > 65); got {len(got)}"
    assert value("third_row")["ts"] == _df0.iloc[2]["ts"], "third_row should be df.iloc[2]"
TESTS['2.1'] = _test_2_1

def _test_2_2():
    assert (value("n_rows"), value("n_cols")) == _raw0.shape, f"shape should be {_raw0.shape}"
    m = value("missing_per_col")
    assert int(m["temp_c"]) == int(_raw0["temp_c"].isna().sum()), "missing_per_col['temp_c'] looks wrong: use df_raw.isna().sum()"
    assert int(value("n_dups")) == 2, f"n_dups should be 2 (duplicates on asset_id + ts), got {n_dups}"
    assert len(value("df_clean")) == len(_raw0) - 2, "df_clean should have the 2 duplicate rows removed"
TESTS['2.2'] = _test_2_2

def _test_2_3():
    _same_df_check()
    got, ref = value("df_filled"), _ref_fill(_df0)
    assert len(got) == len(ref), "df_filled should have the same number of rows as df"
    n_na = int(got["temp_c"].isna().sum())
    if n_na == int(_df0["temp_c"].isna().sum()):
        raise NotImplementedError
    assert n_na == 6, f"exactly 6 values (the 6-hour gap on T1) should stay NaN; you have {n_na}"
    assert np.allclose(got["temp_c"].fillna(-999).values, ref["temp_c"].fillna(-999).values, atol=1e-6), "filled values don't match linear interpolation within each asset"
TESTS['2.3'] = _test_2_3

def _test_2_4():
    s = value("summary")
    if "asset_id" in getattr(s, "columns", []):
        s = s.set_index("asset_id")
    for col in ["mean_temp", "max_temp", "n"]:
        assert col in s.columns, f"summary needs a column named {col}"
    ref = _df0.groupby("asset_id").agg(mean_temp=("temp_c", "mean"), max_temp=("temp_c", "max"), n=("temp_c", "size"))
    assert np.allclose(s.loc[ref.index, ["mean_temp", "max_temp"]].values, ref[["mean_temp", "max_temp"]].values), "mean/max don't match"
    assert list(s.loc[ref.index, "n"]) == [336, 336, 336], "n should count all rows (use 'size', not 'count')"
TESTS['2.4'] = _test_2_4

def _test_2_5():
    d = value("daily")
    assert {"asset_id", "ts", "load_kw"} <= set(d.columns), "daily needs columns asset_id, ts, load_kw (use .reset_index())"
    assert len(d) == 42, f"daily should have 42 rows, got {len(d)}"
    ref = _df0.set_index("ts").groupby("asset_id")["load_kw"].resample("D").mean().reset_index()
    a = d.sort_values(["asset_id", "ts"])["load_kw"].values
    assert np.allclose(a, ref.sort_values(["asset_id", "ts"])["load_kw"].values), "daily averages don't match. Did you use mean (not sum)?"
TESTS['2.5'] = _test_2_5

def _test_2_6():
    _same_df_check()
    a = value("anomalies")
    assert "z" in need("df_z").columns, "df_z needs a column z"
    hit = a[(a["asset_id"] == "T2") & (a["ts"] == pd.Timestamp("2026-01-10 14:00"))]
    assert len(hit) == 1, "the 95 °C spike on T2 at 2026-01-10 14:00 should be in anomalies"
    assert len(a) < 10, f"{len(a)} anomalies is too many. Check that the window is per asset and shifted by 1."
TESTS['2.6'] = _test_2_6

def _test_2_7():
    l = value("lab")
    assert "fail_within_48h" in l.columns, "lab needs a column fail_within_48h"
    counts = l.groupby("asset_id")["fail_within_48h"].sum().to_dict()
    assert int(counts.get("T1", 0)) == 49 and int(counts.get("T2", 0)) == 49 and int(counts.get("T3", 0)) == 0, f"positives per asset should be T1 49, T2 49, T3 0; got {counts}"
TESTS['2.7'] = _test_2_7

def _test_2_8():
    w = value("wide")
    assert w.shape == (336, 3), f"wide should be 336 × 3, got {w.shape}"
    assert sorted(map(str, w.columns)) == ["T1", "T2", "T3"], "columns should be the asset IDs"
    c = value("corr")
    assert c.shape == (3, 3) and np.allclose(np.diag(c.values), 1), "corr should be the 3 × 3 correlation matrix"
TESTS['2.8'] = _test_2_8

def _test_2_9():
    _same_df_check()
    d = value("df_status")
    if d["status"].iloc[0] is Ellipsis:
        raise NotImplementedError
    ref = pd.Series(_ref_status(_df0)).value_counts()
    got = value("status_counts")
    for k in ["ok", "warning", "critical"]:
        assert int(got.get(k, 0)) == int(ref.get(k, 0)), f"count for {k!r} should be {int(ref.get(k, 0))}, got {int(got.get(k, 0))}"
TESTS['2.9'] = _test_2_9

def _test_2_10():
    lj = value("left_join")
    assert len(lj) == 7, f"left_join should have 7 rows (T1×2, T2 with no orders, T3×3, T4 with no orders); got {len(lj)}"
    assert int(lj["cost"].isna().sum()) == 2, "T2 and T4 have no work orders, so their cost should be NaN"
    assert list(value("orphans")["asset_id"]) == ["T9"], "orphans should be the single work order for unknown asset T9"
TESTS['2.10'] = _test_2_10

def _test_2_11():
    b = value("busy")
    assert sorted(b["asset_id"]) == ["T1", "T3"], f"busy should be T1 and T3, got {sorted(b['asset_id'])}"
    assert dict(zip(b["asset_id"], b["total"])) == {"T1": 1200, "T3": 1450}, "totals should be T1 1200 and T3 1450"
    l = value("latest")
    assert dict(zip(l["asset_id"], l["cost"])) == {"T1": 700, "T3": 400, "T9": 250}, "latest should give T1 700, T3 400, T9 250"
TESTS['2.11'] = _test_2_11

def _test_2_12():
    c, s, h = value("counts"), value("share"), value("hottest5")
    assert int(c["ok"]) == 3 and int(c["critical"]) == 1, "counts should be ok 3, warning 2, critical 1"
    assert close(s["ok"], 0.5), "share['ok'] should be 0.5 (use normalize=True)"
    assert len(h) == 5 and close(h["temp_c"].iloc[0], 95.0), "hottest5 should start with the 95 °C spike"
TESTS['2.12'] = _test_2_12

print("✅ Setup complete: 12 exercises ready. Work top to bottom.")

### Exercise 2.1 · Selecting columns and rows
<small>📖 Card 2.1 in the visual guide</small>

Create:
- `temp_col`: the `temp_c` column (a Series)
- `t2_hot`: the rows of `df` where `asset_id` is `"T2"` **and** `temp_c` is above 65
- `third_row`: the row in **position** 2 (the third row) of `df`

<details><summary>🔎 Hint</summary>

Combine conditions with `&` and brackets: `df[(cond1) & (cond2)]`. Position means `iloc`.

</details>

In [ ]:
temp_col = ...
t2_hot = ...
third_row = ...

In [ ]:
check("2.1")

<details><summary>💡 Show a solution</summary>

```python
temp_col = df["temp_c"]
t2_hot = df[(df["asset_id"] == "T2") & (df["temp_c"] > 65)]
third_row = df.iloc[2]
```

</details>

### Exercise 2.2 · Data-quality check
<small>📖 Card 2.2 in the visual guide</small>

Using `df_raw`:
- `n_rows, n_cols`: its shape
- `missing_per_col`: number of missing values per column (a Series)
- `n_dups`: how many rows repeat an earlier (`asset_id`, `ts`) pair
- `df_clean`: `df_raw` with those duplicates removed (keep the first)

In [ ]:
n_rows, n_cols = ..., ...
missing_per_col = ...
n_dups = ...
df_clean = ...

In [ ]:
check("2.2")

<details><summary>💡 Show a solution</summary>

```python
n_rows, n_cols = df_raw.shape
missing_per_col = df_raw.isna().sum()
n_dups = df_raw.duplicated(["asset_id", "ts"]).sum()
df_clean = df_raw.drop_duplicates(["asset_id", "ts"])
```

</details>

### Exercise 2.3 · Fill short gaps only, per asset
<small>📖 Card 2.3 in the visual guide</small>

Create `df_filled`: a copy of `df` where missing `temp_c` values are filled by **linear interpolation within each asset**,
but **only for gaps of 3 hours or less**. Longer gaps must stay NaN. Keep the same rows in the same order.

⚠️ `interpolate(limit=3)` is *not* enough: it partly fills long gaps too.

<details><summary>🔎 Hint</summary>

1. `is_na = df["temp_c"].isna()`
2. Give each gap an ID: `run_id = df["temp_c"].notna().groupby(df["asset_id"]).cumsum()`
3. Gap length: `is_na.groupby([df["asset_id"], run_id]).transform("sum")`
4. Interpolate per asset with `groupby(...).transform(lambda s: s.interpolate(limit_area="inside"))`, then `.where(...)` to put NaN back on long gaps.

</details>

In [ ]:
df_filled = df.copy()
...

In [ ]:
check("2.3")

<details><summary>💡 Show a solution</summary>

```python
df_filled = df.copy()
is_na = df_filled["temp_c"].isna()
run_id = df_filled["temp_c"].notna().groupby(df_filled["asset_id"]).cumsum()
gap_len = is_na.groupby([df_filled["asset_id"], run_id]).transform("sum")
filled = df_filled.groupby("asset_id")["temp_c"].transform(lambda s: s.interpolate(limit_area="inside"))
df_filled["temp_c"] = filled.where(~(is_na & (gap_len > 3)))
```

</details>

### 🗣 Say it out loud · 2.3s

**“Why wouldn't you just fill every gap?”**

*Answer out loud first, or type your answer in the cell below. Then compare.*

✍️ *Your answer (double-click to edit):*



<details><summary>💬 Show a model answer</summary>

A short gap of an hour or two is usually a dropped message, and a straight line across it is a reasonable estimate. A long gap often means the sensor or the asset was actually down. Filling it would invent hours of data that never happened, hide a real outage from the operations team, and make the model trust fake values. So I fill only short gaps, per asset, and flag long gaps explicitly.

</details>

### Exercise 2.4 · groupby summary
<small>📖 Card 2.4 in the visual guide</small>

Create `summary`: one row per `asset_id` with columns `mean_temp`, `max_temp` (from `temp_c`) and `n` (number of rows, including NaN).

In [ ]:
summary = ...

In [ ]:
check("2.4")

<details><summary>💡 Show a solution</summary>

```python
summary = df.groupby("asset_id").agg(
    mean_temp=("temp_c", "mean"),
    max_temp=("temp_c", "max"),
    n=("temp_c", "size"),
)
```

</details>

### Exercise 2.5 · Hourly → daily
<small>📖 Card 2.5 in the visual guide</small>

Create `daily`: the **average** `load_kw` per asset per day, as a normal DataFrame with columns `asset_id`, `ts`, `load_kw` (42 rows: 3 assets × 14 days).

In [ ]:
daily = ...

In [ ]:
check("2.5")

<details><summary>💡 Show a solution</summary>

```python
daily = (df.set_index("ts")
           .groupby("asset_id")["load_kw"]
           .resample("D").mean()
           .reset_index())
```

</details>

### Exercise 2.6 · Rolling z-score anomalies
<small>📖 Card 2.6 in the visual guide</small>

Create `df_z`: a copy of `df` with a column `z`, the rolling z-score of `temp_c` **per asset**, using the **previous** 24 hours
(window 24, `min_periods=12`, then `.shift(1)`). Then create `anomalies`: the rows of `df_z` where `|z| > 4`.

In [ ]:
df_z = df.copy()
...
anomalies = ...

In [ ]:
check("2.6")

<details><summary>💡 Show a solution</summary>

```python
df_z = df.copy()
g = df_z.groupby("asset_id")["temp_c"]
roll_mean = g.transform(lambda s: s.rolling(24, min_periods=12).mean().shift(1))
roll_std = g.transform(lambda s: s.rolling(24, min_periods=12).std().shift(1))
df_z["z"] = (df_z["temp_c"] - roll_mean) / roll_std
anomalies = df_z[df_z["z"].abs() > 4]
```

</details>

### Exercise 2.7 · Failure labels with merge_asof
<small>📖 Card 2.7 in the visual guide</small>

Create `lab`: for every row of `df` (just `asset_id` and `ts` are needed), find the **next** failure of the same asset using `pd.merge_asof`,
and add a boolean column `fail_within_48h` that is True when that failure is at most 48 hours after the reading.

<details><summary>🔎 Hint</summary>

Both tables must be sorted by their time column. Use `left_on="ts", right_on="fail_ts", by="asset_id", direction="forward"`.

</details>

In [ ]:
lab = ...

In [ ]:
check("2.7")

<details><summary>💡 Show a solution</summary>

```python
lab = pd.merge_asof(
    df[["asset_id", "ts"]].sort_values("ts"),
    failures.sort_values("fail_ts"),
    left_on="ts", right_on="fail_ts", by="asset_id", direction="forward",
)
lab["fail_within_48h"] = (lab["fail_ts"] - lab["ts"]) <= pd.Timedelta("48h")
```

</details>

### Exercise 2.8 · Long to wide
<small>📖 Card 2.8 in the visual guide</small>

Create `wide`: one row per `ts`, one column per asset, values = `load_kw`. Then `corr`: the correlation matrix of `wide`.

In [ ]:
wide = ...
corr = ...

In [ ]:
check("2.8")

<details><summary>💡 Show a solution</summary>

```python
wide = df.pivot_table(index="ts", columns="asset_id", values="load_kw")
corr = wide.corr()
```

</details>

### Exercise 2.9 · Label every row without apply
<small>📖 Card 2.9 in the visual guide</small>

Create `df_status`: a copy of `df` with a column `status` = `"critical"` if `temp_c` > 80, `"warning"` if > 65, else `"ok"`. Use `np.select`.
Then `status_counts`: how many rows have each status.

In [ ]:
df_status = df.copy()
df_status["status"] = ...
status_counts = ...

In [ ]:
check("2.9")

<details><summary>💡 Show a solution</summary>

```python
df_status = df.copy()
df_status["status"] = np.select(
    [df_status["temp_c"] > 80, df_status["temp_c"] > 65],
    ["critical", "warning"],
    default="ok",
)
status_counts = df_status["status"].value_counts()
```

</details>

### Exercise 2.10 · Joins
<small>📖 Card 2.10 in the visual guide</small>

- `left_join`: every asset with its work orders (`assets` on the left, join on `asset_id`)
- `orphans`: the work orders whose `asset_id` is **not** in `assets`

<details><summary>🔎 Hint</summary>

`work_orders.merge(assets, on="asset_id", how="left", indicator=True)` adds a `_merge` column; keep rows where it is `"left_only"`.

</details>

In [ ]:
left_join = ...
orphans = ...

In [ ]:
check("2.10")

<details><summary>💡 Show a solution</summary>

```python
left_join = assets.merge(work_orders, on="asset_id", how="left")
m = work_orders.merge(assets, on="asset_id", how="left", indicator=True)
orphans = m[m["_merge"] == "left_only"]
```

</details>

### 🗣 Say it out loud · 2.10s

**“After a left merge, your table has more rows than the left table had. What happened, and what do you do?”**

*Answer out loud first, or type your answer in the cell below. Then compare.*

✍️ *Your answer (double-click to edit):*



<details><summary>💬 Show a model answer</summary>

Some keys on the left match several rows on the right. For example, one asset has several work orders, so its row is repeated once per match. That's expected for a one-to-many relationship, but it can also be a duplicate-key bug in the right table. I check with `validate="one_to_many"` (or `"one_to_one"` if each key should match once), look at `value_counts()` of the key, and always compare row counts before and after a merge.

</details>

### Exercise 2.11 · SQL-style questions in pandas
<small>📖 Card 2.11 in the visual guide</small>

- `busy`: assets with **2 or more** work orders, with columns `asset_id`, `n` (count) and `total` (sum of cost). Like SQL `GROUP BY … HAVING`.
- `latest`: the **most recent** work order per asset (one row per asset, all columns). Like `ROW_NUMBER() … WHERE rn = 1`.

In [ ]:
busy = ...
latest = ...

In [ ]:
check("2.11")

<details><summary>💡 Show a solution</summary>

```python
g = work_orders.groupby("asset_id").agg(n=("cost", "size"), total=("cost", "sum")).reset_index()
busy = g[g["n"] >= 2]
latest = work_orders.sort_values("date").groupby("asset_id").tail(1)
```

</details>

### Exercise 2.12 · value_counts and nlargest
<small>📖 Card 2.12 in the visual guide</small>

Given `codes` below:
- `counts`: how many times each status appears
- `share`: the same as proportions (0–1)
- `hottest5`: the 5 rows of `df` with the highest `temp_c`

In [ ]:
codes = pd.Series(["ok", "ok", "warning", "ok", "critical", "warning"])
counts = ...
share = ...
hottest5 = ...

In [ ]:
check("2.12")

<details><summary>💡 Show a solution</summary>

```python
codes = pd.Series(["ok", "ok", "warning", "ok", "critical", "warning"])
counts = codes.value_counts()
share = codes.value_counts(normalize=True)
hottest5 = df.nlargest(5, "temp_c")
```

</details>

## 🏁 Your progress
Run the cell below to see which exercises you've passed.

In [ ]:
progress()